# Machine Learning Midterm Project
## Notebook Template (Country X Synthetic Student Scholarship Dataset)

**Team:** (Name1 - ID1), (Name2 - ID2)

**Goal:** Build an end-to-end classical ML pipeline to predict `Scholarship`.

**Allowed libraries:** `pandas, numpy, matplotlib, seaborn, scikit-learn`

> **Important:** Your grade is primarily based on `report.pdf`. This notebook is for reproducibility and spot-checking.


# 0. Setup
Run this cell first. Do not add extra non-allowed libraries.


In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

# Reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)


# 1. Load Dataset
Place the CSV in the same folder as this notebook, or provide a path.


In [ ]:
# TODO: update filename if needed
DATA_PATH = "synthetic_student_scholarship_train.csv"  # provided by instructor

df = pd.read_csv(DATA_PATH)
df.head()

# 2. Quick Dataset Overview (EDA)
Fill your report with insights, not just plots.


In [ ]:
print("Shape:", df.shape)
df.info()

In [ ]:
df.describe(include='all').T.head(20)

## 2.1 Missing Values
Identify which columns have missing values and the approximate rates.


In [ ]:
missing = df.isna().mean().sort_values(ascending=False)
missing[missing > 0]

## 2.2 Target Distribution
Check class balance and consider which metrics are appropriate.


In [ ]:
target_col = "Scholarship"
df[target_col].value_counts(dropna=False)

In [ ]:
sns.countplot(x=target_col, data=df)
plt.title("Target distribution")
plt.show()

## 2.3 Example Visualizations (minimum 4)
Replace/extend these plots. In your report, every figure must have a caption + interpretation.


In [ ]:
plt.figure(figsize=(6,4))
sns.histplot(df["GPA"], kde=True)
plt.title("Distribution of GPA")
plt.show()

In [ ]:
plt.figure(figsize=(6,4))
sns.boxplot(x=target_col, y="AttendanceRate", data=df)
plt.title("AttendanceRate vs Scholarship")
plt.show()

In [ ]:
# Correlation heatmap (numeric only)
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
plt.figure(figsize=(10,6))
sns.heatmap(df[num_cols].corr(), annot=False)
plt.title("Correlation heatmap (numeric features)")
plt.show()

# 3. Train/Test Split
Use a fixed seed for reproducibility.


In [ ]:
X = df.drop(columns=[target_col])
y = df[target_col].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_SEED,
    stratify=y
)

X_train.shape, X_test.shape

# 4. Preprocessing Pipeline
Use `ColumnTransformer` + `Pipeline` to avoid leakage and keep code clean.


In [ ]:
# Identify column types
numeric_features = X_train.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=[np.number]).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

numeric_features, categorical_features

# 5. Train Required Models
Required: Logistic Regression, Decision Tree, Random Forest.


In [ ]:
models = {
    "LogisticRegression": LogisticRegression(max_iter=1000, random_state=RANDOM_SEED),
    "DecisionTree": DecisionTreeClassifier(random_state=RANDOM_SEED),
    "RandomForest": RandomForestClassifier(
        n_estimators=300, random_state=RANDOM_SEED, n_jobs=-1
    )
}

def evaluate_model(name, pipe, X_te, y_te):
    y_pred = pipe.predict(X_te)
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_te, y_pred),
        "Precision": precision_score(y_te, y_pred, zero_division=0),
        "Recall": recall_score(y_te, y_pred, zero_division=0),
        "F1": f1_score(y_te, y_pred, zero_division=0),
        "ConfusionMatrix": confusion_matrix(y_te, y_pred)
    }

results = []
trained_pipes = {}

for name, clf in models.items():
    pipe = Pipeline(steps=[("preprocess", preprocessor), ("model", clf)])
    pipe.fit(X_train, y_train)
    trained_pipes[name] = pipe
    results.append(evaluate_model(name, pipe, X_test, y_test))

results_df = pd.DataFrame([{k:v for k,v in r.items() if k != "ConfusionMatrix"} for r in results])
results_df.sort_values(by="F1", ascending=False)

## 5.1 Confusion Matrix (Best Model)
Pick the best model based on your chosen metric(s) and show confusion matrix + interpretation.


In [ ]:
# TODO: choose best model name
best_model_name = results_df.sort_values(by="F1", ascending=False).iloc[0]["Model"]
best_cm = [r["ConfusionMatrix"] for r in results if r["Model"] == best_model_name][0]

print("Best model:", best_model_name)
print(best_cm)

plt.figure(figsize=(4,3))
sns.heatmap(best_cm, annot=True, fmt="d")
plt.title(f"Confusion Matrix: {best_model_name}")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()

# 6. Feature Importance / Interpretability (Recommended for Report)
For Random Forest: feature importances are straightforward. For Logistic Regression: coefficients.


In [ ]:
# Example: Random Forest feature importances (after preprocessing)
rf_pipe = trained_pipes.get("RandomForest")
if rf_pipe is not None:
    ohe = rf_pipe.named_steps["preprocess"].named_transformers_["cat"].named_steps["onehot"]
    cat_feature_names = ohe.get_feature_names_out(categorical_features).tolist()
    feature_names = numeric_features + cat_feature_names

    importances = rf_pipe.named_steps["model"].feature_importances_
    imp_df = pd.DataFrame({"feature": feature_names, "importance": importances})
    imp_df = imp_df.sort_values(by="importance", ascending=False).head(15)
    display(imp_df)

    plt.figure(figsize=(7,4))
    sns.barplot(x="importance", y="feature", data=imp_df)
    plt.title("Top-15 Feature Importances (Random Forest)")
    plt.tight_layout()
    plt.show()


# 7. (Optional) Generate Predictions for Hidden Test
If the instructor uses a hidden test set, you may be asked to generate `predictions.csv`.

**Format requirement:** two columns: `StudentID,ScholarshipPred` with `ScholarshipPred` in {0,1}.


In [ ]:
# TODO: update path if instructor provides hidden test file
# HIDDEN_PATH = "synthetic_student_scholarship_hidden_test.csv"
# hidden = pd.read_csv(HIDDEN_PATH)
#
# best_pipe = trained_pipes[best_model_name]
# pred = best_pipe.predict(hidden)
#
# out = pd.DataFrame({"StudentID": hidden["StudentID"].astype(int), "ScholarshipPred": pred.astype(int)})
# out.to_csv("predictions.csv", index=False)
# out.head()

# 8. Final Notes
- Make sure your report includes **insights** (not just plots).
- Explain your preprocessing choices and leakage prevention.
- Compare models fairly and justify your best model.
